# Hadoop HDFS and YARN Sandbox

Explora HDFS, la alta disponibilidad y YARN con el dataset de medidores. Ejecuta primero `python lessons/00_seed.py`. Las celdas de experimentación usan `/data/work/sandbox` para no modificar los datos de origen.

## 0. Conectar con el clúster

In [ ]:
import pandas as pd
from lab.config import RAW, WORK
from lab.hdfs import datanodes, fsck_blocks, hdfs, namenode_states, queues, yarn_nodes

pd.set_option('display.max_colwidth', 120)
print(hdfs('getconf', '-confKey', 'fs.defaultFS'))
pd.DataFrame(namenode_states())

## 1. HDFS: observar bloques y réplicas
Cambia el tamaño de bloque y la replicación para comparar la ubicación de los bloques.

In [ ]:
DAY = f'{RAW}/readings/readings_2026-09-01.csv'
DEMO = f'{WORK}/sandbox'
hdfs('dfs', '-mkdir', '-p', DEMO)
hdfs('dfs', '-D', 'dfs.blocksize=1m', '-D', 'dfs.replication=3', '-cp', '-f', DAY, f'{DEMO}/day.csv')
hdfs('dfs', '-setrep', '-w', '3', f'{DEMO}/day.csv')
pd.DataFrame(fsck_blocks(f'{DEMO}/day.csv'))

In [ ]:
# Juega aquí: cambia la replicación y revisa cuánto tarda la re-replicación.
hdfs('dfs', '-setrep', '2', f'{DEMO}/day.csv')
pd.DataFrame(fsck_blocks(f'{DEMO}/day.csv'))

## 2. Metadatos y rack awareness

In [ ]:
pd.DataFrame(datanodes())
print(hdfs('dfsadmin', '-printTopology'))
print(hdfs('dfs', '-count', '-q', '-h', '-v', RAW))

## 3. YARN: recursos y colas

In [ ]:
pd.DataFrame(yarn_nodes())

In [ ]:
# Juega aquí: compara las capacidades garantizadas de prod y dev.
pd.DataFrame(queues())

## 4. Un MapReduce de ejemplo
La lección 07 muestra los pasos del mapper, combiner, shuffle, particionador y reducer con contadores reales de YARN.

In [ ]:
from lab.mapreduce import streaming

# Inspecciona la salida ya escrita por la lección 07.
OUT = f'{WORK}/07/trend'
print(hdfs('dfs', '-ls', '-h', OUT))
print(hdfs('dfs', '-cat', f'{OUT}/part-00000')[:1200])

## 5. Limpieza opcional
Solo borra los archivos creados en `/data/work/sandbox`.

In [ ]:
# hdfs('dfs', '-rm', '-r', '-skipTrash', DEMO)